In [1]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [2]:
y = df["Churn"].map({
    "No" : 0,
    "Yes": 1
})

In [3]:
X = df.drop(columns=['customerID', 'Churn'])

In [4]:
X.shape

(7043, 19)

In [5]:
y.shape

(7043,)

In [6]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

## split into training and test data

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
X_train.shape, X_test.shape

((5634, 19), (1409, 19))

In [11]:
y_train.value_counts(normalize=True) * 100

Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

In [12]:
y_test.value_counts(normalize=True)*100

Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64

## idenitfy what needs preprocessing

In [13]:
X_train.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object

In [14]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    col for col in X_train.columns
    if col not in numerical_features + ['SeniorCitizen']
]  # used list comprehension

len(categorical_features) # expect 19-3num features - 1 sr citizen = 15

15

## create the preprocessor


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
preprocessor = ColumnTransformer(
    transformers = [
        ('nums', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

## build the ML pipeline

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [23]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [24]:
# run the model
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nums', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, defa

## Train the first model


In [25]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nums', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

## make predictions on unseen customers

In [26]:
y_pred = model.predict(X_test)


In [27]:
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

results.head(10)

,Actual,Predicted
437,0,0
2280,0,1
2235,0,0
4460,0,0
3761,0,0
5748,0,1
3568,0,0
2976,0,0
5928,0,0
1639,1,0


## build the confusion matrix

In [28]:
from sklearn.metrics import confusion_matrix

cm=confusion_matrix(y_test, y_pred)
cm

array([[926, 109],
       [165, 209]])

In [29]:
209 / (209 + 165)

0.5588235294117647

In [30]:
209 / (209 + 109)

0.6572327044025157

In [31]:
2*(0.657*0.559)/(0.657+0.559)

0.6040509868421052

In [33]:
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print('Accuracy: ', accuracy_score(y_test, y_pred))
print('Precision: ',precision_score(y_test, y_pred))
print('Recall: ',recall_score(y_test, y_pred))
print('F1: ',f1_score(y_test, y_pred))

Accuracy:  0.8055358410220014
Precision:  0.6572327044025157
Recall:  0.5588235294117647
F1:  0.6040462427745664


So the baseline Logistic Regression currently looks roughly like:

| Metric | Score | Meaning |
|---|---:|---|
| Precision | **65.7%** | When we predict churn, we're right about 66% of the time |
| Recall | **55.9%** | We catch about 56% of actual churners |
| F1 | **60.4%** | Balance between precision and recall |

In [36]:
# underneath, Logistic Regression first estimates a probability.
y_probability = model.predict_proba(X_test)

y_probability[:5]

array([[0.95427494, 0.04572506],
       [0.3163454 , 0.6836546 ],
       [0.9413611 , 0.0586389 ],
       [0.59774945, 0.40225055],
       [0.9784311 , 0.0215689 ]])

In [38]:
churn_probability = model.predict_proba(X_test)[:,1]

churn_probability[:10]

array([0.04572506, 0.6836546 , 0.0586389 , 0.40225055, 0.0215689 ,
       0.60323945, 0.44836685, 0.12995842, 0.00285883, 0.3942891 ])

By default, the classifier effectively uses a threshold around: 0.50

And this creates a major business decision:
What if catching more churners matters enough that we're willing to tolerate more false alarms?

In [39]:
## lower the threshold of 0.40
y_pred_040 = (churn_probability >= 0.40).astype(int)

print('Precision: ', precision_score(y_test, y_pred_040))
print('Recall: ', recall_score(y_test, y_pred_040))
print('F1: ', f1_score(y_test, y_pred_040))




Precision:  0.5681818181818182
Recall:  0.6684491978609626
F1:  0.6142506142506142


| Threshold | Precision | Recall | F1 |
|---|---:|---:|---:|
| **0.50** | 65.7% | 55.9% | 60.4% |
| **0.40** | 56.8% ↓ | **66.8% ↑** | **61.4% ↑** |

In [47]:
## compare several threshold

threshold = [0.30, 0.40, 0.50, 0.60, 0.70]

for th in threshold:
    predictions = (churn_probability >= th).astype(int)
    
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    
    print(
        f"Threshold: {th:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.30 | Precision: 0.519 | Recall: 0.754 | F1: 0.615
Threshold: 0.40 | Precision: 0.568 | Recall: 0.668 | F1: 0.614
Threshold: 0.50 | Precision: 0.657 | Recall: 0.559 | F1: 0.604
Threshold: 0.60 | Precision: 0.718 | Recall: 0.401 | F1: 0.515
Threshold: 0.70 | Precision: 0.742 | Recall: 0.184 | F1: 0.296


## Run 5-fold statified cross-validation
Because churn is imbalanced, we'll use StratifiedKFold to preserve the churn/non-churn proportions within each fold

In [48]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [49]:
# create the 5 fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [50]:
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['precision', 'recall', 'f1']
)

In [51]:
cv_results['test_f1']

array([0.58608059, 0.54945055, 0.58888889, 0.64248705, 0.59848485])

In [52]:
cv_results['test_f1'].mean()

np.float64(0.5930783839073995)

In [53]:
cv_results["test_recall"].mean()

np.float64(0.5438127090301004)

In [54]:
cv_results["test_precision"].mean()

np.float64(0.6538107624505562)

5-fold validation results
| Metric | Mean |
|---|---:|
| Precision | **65.4%** |
| Recall | **54.4%** |
| F1 | **59.3%** |

In [55]:
from sklearn.model_selection import cross_val_predict

In [56]:
cv_probabilities = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba'
)[:, 1]

In [57]:
cv_probabilities.shape

(5634,)

## choose a threshold using cross-validation predictions

In [58]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

for threshold in thresholds:
    cv_predictions = (cv_probabilities >= threshold).astype(int)

    precision = precision_score(y_train, cv_predictions)
    recall = recall_score(y_train, cv_predictions)
    f1 = f1_score(y_train, cv_predictions)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.30 | Precision: 0.541 | Recall: 0.767 | F1: 0.635
Threshold: 0.40 | Precision: 0.594 | Recall: 0.664 | F1: 0.627
Threshold: 0.50 | Precision: 0.654 | Recall: 0.544 | F1: 0.594
Threshold: 0.60 | Precision: 0.710 | Recall: 0.397 | F1: 0.509
Threshold: 0.70 | Precision: 0.782 | Recall: 0.211 | F1: 0.333


In [59]:
final_threshold = 0.30

In [60]:
test_probabilities = model.predict_proba(X_test)[:, 1]

In [64]:
# apply the threshold selected 0.30
final_predictions =(
    test_probabilities >= final_threshold
).astype(int)

In [65]:
print("Precision:", precision_score(y_test, final_predictions))
print("Recall:", recall_score(y_test, final_predictions))
print("F1:", f1_score(y_test, final_predictions))

Precision: 0.5193370165745856
Recall: 0.7540106951871658
F1: 0.6150490730643402


In [66]:
confusion_matrix(y_test, final_predictions)

array([[774, 261],
       [ 92, 282]])

The model identified about three-quarters of customers who actually churned. This higher recall comes with more false alarms: 261 customers were flagged for retention even though they ultimately stayed.

move from:
How well does the model predict?

to:
What patterns did the model learn?

In [67]:
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

In [68]:
coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

In [69]:
coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

In [70]:
coefficient_df = coefficient_df.sort_values(
    "Coefficient",
    ascending=False
)

coefficient_df

,Feature,Coefficient
15,cat__InternetService_Fiber optic,0.651066
35,cat__Contract_Month-to-month,0.576117
2,nums__TotalCharges,0.525363
31,cat__StreamingTV_Yes,0.208361
34,cat__StreamingMovies_Yes,0.208304
42,cat__PaymentMethod_Electronic check,0.197795
17,cat__OnlineSecurity_No,0.156808
44,remainder__SeniorCitizen,0.144545
26,cat__TechSupport_No,0.131505
13,cat__MultipleLines_Yes,0.109912


In [71]:
coefficient_df.head(5)

,Feature,Coefficient
15,cat__InternetService_Fiber optic,0.651066
35,cat__Contract_Month-to-month,0.576117
2,nums__TotalCharges,0.525363
31,cat__StreamingTV_Yes,0.208361
34,cat__StreamingMovies_Yes,0.208304


In [72]:
coefficient_df.tail(5)

,Feature,Coefficient
38,cat__PaperlessBilling_No,-0.343703
1,nums__MonthlyCharges,-0.620723
14,cat__InternetService_DSL,-0.660859
37,cat__Contract_Two year,-0.770715
0,nums__tenure,-1.249501


In [73]:
import numpy as np

coefficient_df["OddsRatio"] = np.exp(
    coefficient_df["Coefficient"]
)

In [74]:
coefficient_df[
    coefficient_df["Feature"].isin([
        "cat__Contract_Month-to-month",
        "cat__Contract_Two year",
        "nums__tenure"
    ])
]

,Feature,Coefficient,OddsRatio
35,cat__Contract_Month-to-month,0.576117,1.779116
37,cat__Contract_Two year,-0.770715,0.462682
0,nums__tenure,-1.249501,0.286648


In [76]:
import joblib

joblib.dump(
    model,
    "../models/churn_model.joblib"
)

['../models/churn_model.joblib']